# PAS SigLIP DEFT Pipeline

This notebook will showcase DEFT for Person Attribute Search (PAS).

* This notebook assumes certain prerequisites are already installed on the host. The setup phase does not install these — it installs Python libraries, pulls the TAO Docker images, and prepares the datasets. It **must** be run first.
* Zero-shot evaluation will run to establish a baseline
* The DEFT loop will be run

## <a name="Prerequisites"></a>Prerequisites

This notebook does **not** install any system-level prerequisites — pip, Docker, an NVIDIA driver, or the
NVIDIA Container Toolkit. They must already be present on the host before running the setup cells below, or
the first cells will fail with `command not found`.

See the repository [README's Software requirements](../../../README.md#Softwarerequirements) table for the
required software and versions.

## Setup

### Local code
Alongside this notebook is a library called `pas_deft`, which includes several important functions that
will be used in the DEFT loop.

Run the following cell to install the package in editable-compatible form 
and pull in all required libraries
(`pyyaml`, `pandas`, `pyarrow`, `numpy`, `scikit-learn`, `matplotlib`, `Pillow`, `toml`, `torch`).

In [ ]:
!pip install --upgrade pip
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install -e pas_deft/

### Docker

Certain steps must be run within the TAO Docker containers. To ensure you have the correct versions, run the following:

In [ ]:
import getpass
import os
import subprocess

tao_pyt_image = "nvcr.io/nvidia/tao/tao-toolkit:7.1.0-pyt"
tao_ds_image = "nvcr.io/nvidia/tao/tao-toolkit:7.1.0-data-services"


def run_container(cmd: str) -> None:
    """Run a container command, printing it and raising if it fails.

    A stage in the DEFT loop that fails must stop the round instead of
    letting the next stage silently run against stale/missing outputs.
    """
    print(cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


run_container(f"docker pull {tao_pyt_image}")
run_container(f"docker pull {tao_ds_image}")

HOST_RESULTS_DIR = os.path.abspath("results")
HOST_DATA_DIR = os.path.abspath("data")
HOST_SPECS_DIR = os.path.abspath("specs")
HOST_PATCHES_DIR = os.path.abspath("patches")
HOST_CACHE_DIR = os.path.abspath("cache")

os.makedirs(HOST_RESULTS_DIR, exist_ok=True)
os.makedirs(HOST_DATA_DIR, exist_ok=True)
os.makedirs(HOST_CACHE_DIR, exist_ok=True)

DOCKER_CMD = (
    "docker run --gpus all --rm --ipc=host "
    f"--user {os.getuid()}:{os.getgid()} "
    f"-e USER={getpass.getuser()} -e LOGNAME={getpass.getuser()} "
    f"-e HOME=/tmp "
    f"-e PYTHONPATH=/patches "
    f"-e HF_HOME=/cache/huggingface -e XDG_CACHE_HOME=/cache "
    f"-v {HOST_RESULTS_DIR}:/results "
    f"-v {HOST_DATA_DIR}:/data "
    f"-v {HOST_DATA_DIR}:{HOST_DATA_DIR} "
    f"-v {HOST_SPECS_DIR}:/specs "
    f"-v {HOST_PATCHES_DIR}:/patches:ro "
    f"-v {HOST_CACHE_DIR}:/cache"
)

**Using a different location for results/data/specs/etc.**

By default, `HOST_RESULTS_DIR`, `HOST_DATA_DIR`, `HOST_SPECS_DIR`, `HOST_PATCHES_DIR`, and `HOST_CACHE_DIR` above resolve to `results/`, `data/`, `specs/`, `patches/`, and `cache/` next to this notebook. If you want any of these to live somewhere else (e.g. a larger disk), just point the corresponding `HOST_*_DIR` variable at that path instead, for example:

```python
HOST_RESULTS_DIR = "/mnt/big_disk/pas_results"
```

The `-v {HOST_X_DIR}:/x` entries in `DOCKER_CMD` mount your chosen host directory to a fixed container-side path (`/results`, `/data`, `/specs`, `/patches`, `/cache`). The experiment YAMLs (e.g. `specs/deft_config.yaml`) are read *inside* the container, so any path written into them must be the **container-side** path (e.g. `/data/...`, `/results/...`).

### Dataset

3 files are necessary for the datasets to get constructed:

* `images_raw.tar`
* `meta.tar.gz`
* `SHA256SUMS`

Place them under the `data/` directory. Then, run the following cell to set up the training and evaluation datasets, as well as the mining pool.

In [ ]:
%%bash
set -euo pipefail
cd data
mkdir -p pas_v31_tao_ft
for f in meta.tar.gz images_raw.tar SHA256SUMS; do
    if [ -f "$f" ]; then
        mv "$f" pas_v31_tao_ft/
    fi
done
cd pas_v31_tao_ft

if command -v sha256sum >/dev/null 2>&1; then
    sha256sum -c SHA256SUMS
else
    shasum -a 256 -c SHA256SUMS
fi

tar -xzf meta.tar.gz
tar -xf images_raw.tar
python3 rebuild.py --workers 16


### Imports

In [ ]:
from pas_deft.analyze_gaps import analyze_clip_inference_gaps
from pas_deft.config import PasDeftConfig, bool_str
from pas_deft.data_mining import (
    convert_clip_image_list_to_parquet,
    convert_mined_parquet_to_clip_image_list,
    materialize_pas_eval_split,
    materialize_pas_pool_split,
    materialize_pas_training_split,
    summarize_knn_mining,
    track_cumulative_mined_unique_names,
    write_iteration_summary,
)
from pas_deft.history_aware_mining import (
    select_history_aware_mined_pairs,
)
from pas_deft.utils import (
    create_clip_eval_config,
    create_clip_train_config,
    get_current_checkpoint,
    normalize_clip_pretrained_checkpoint,
    resolve_prev_clip_train_config,
    resolve_prev_eval_dir,
)
from pas_deft.visualization import (
    create_tsne_visualization,
    export_clip_sample_contact_sheets,
    prepare_clip_images_for_embedding,
    prepare_prev_clip_data_for_embedding,
)

## DEFT

1. Run zero-shot evaluation
2. Loop: gap analysis → mine → history-aware selection → visualize → train → eval → iteration summary

### Configuration

All DEFT parameters are defined in `specs/deft_config.yaml`. Feel free to edit this, though the loop should work if all cells were executed in order.

In [ ]:
# === Fill in these values ===

# Path to your pipeline YAML config file (equivalent to --config)
config_path = "specs/deft_config.yaml"

cfg = PasDeftConfig(config_path)

print(cfg)
print(f"  History-aware mining: {cfg.mining.history_aware.enabled}")
print(f"  Visualize           : {cfg.visualization.enabled}")
print(f"  Visualize embeddings: {cfg.visualization.embeddings}")
print(f"  Start iteration     : {cfg.iteration.start}")
print(f"  End iteration       : {cfg.iteration.end}")


This step will set up all the datasets and embed the mining pool. Since this is done separately from the DEFT loop, if you wish to update the mining pool at any point, delete the old `embeddings/source/embeddings.parquet` and re-run this cell.

In [ ]:
eval_image_list_file = f"{cfg.pas_splits_dir}/eval_list.txt"
eval_pairs_file = f"{cfg.pas_splits_dir}/eval_pairs.json"
val_image_list_file = f"{cfg.pas_splits_dir}/val_list.txt"
aug_pool_image_list_file = f"{cfg.pas_splits_dir}/aug_pool_list.txt"
aug_pool_pairs_file = f"{cfg.pas_splits_dir}/aug_pool_pairs.json"

print("\n=== Init: materialising PAS eval split ===", flush=True)
materialize_pas_eval_split(
    eval_pairs_source_file=cfg.pas.eval_pairs_source_file,
    eval_image_list_file=eval_image_list_file,
    eval_pairs_file=eval_pairs_file,
    query_types=cfg.pas.query_types,
    val_image_list_file=val_image_list_file,
    val_sample_size=cfg.pas.val_sample_size,
)

print(f"\n=== Init: materialising PAS pool split ===", flush=True)
materialize_pas_pool_split(
    pool_pairs_source_file=cfg.pas.pool_pairs_source_file,
    aug_pool_image_list_file=aug_pool_image_list_file,
    aug_pool_pairs_file=aug_pool_pairs_file,
    augmented_suffix=cfg.pas.augmented_suffix,
    query_types=cfg.pas.query_types,
    max_aug_pool_rows=cfg.pas.max_aug_pool_rows,
    mining_pool_mode=cfg.pas.mining_pool_mode,
)

# Written once, read by every iteration of the DEFT loop below.
pool_embed_dir = f"{cfg.base_experiment_path}/embeddings/source"
source_pool_pq = f"{pool_embed_dir}/source_pool.parquet"
source_embed_pq = f"{pool_embed_dir}/embeddings.parquet"

print("\n=== Init: converting pool image list to parquet ===", flush=True)
convert_clip_image_list_to_parquet(
    image_list_file=aug_pool_image_list_file,
    image_dir=cfg.pas.source_image_dir,
    output_parquet=source_pool_pq,
    caption_dir=cfg.pas.source_caption_dir,
    caption_file_suffix=".txt",
    pairs_file=aug_pool_pairs_file,
)

if os.path.isfile(source_embed_pq):
    print(f"\n=== Init: pool embeddings already present, skipping: {source_embed_pq} ===", flush=True)
else:
    print("\n=== Init: embedding the mining pool (this takes a while) ===", flush=True)
    run_container(
        f"{DOCKER_CMD} {tao_ds_image} "
        f"embedding text_embeddings -e /specs/text_embed_spec.yaml "
        f"input_parquet={cfg.container_path(source_pool_pq)} "
        f"output_parquet={cfg.container_path(source_embed_pq)}"
    )

### Zero-Shot Evaluation

In [ ]:
zs_dir = f"{cfg.base_experiment_path}/zs"

# ZS Eval
print("\n=== Init: building zero-shot eval config ===", flush=True)
zs_eval_config_path = create_clip_eval_config(
    base_config_yaml=cfg.experiment.eval_config,
    new_config_yaml=f"{zs_dir}/specs/eval_config.yaml",
    output_dir=cfg.container_path(zs_dir),
    checkpoint_path=cfg.training.init_checkpoint,
    eval_image_dir=cfg.pas.eval_image_dir,
    eval_caption_dir=cfg.pas.eval_caption_dir,
    eval_image_list_file=cfg.container_path(eval_image_list_file),
    caption_file_suffix=".txt",
)
print(f"Zero-shot eval config: {zs_eval_config_path}")

print("\n=== Init: zero-shot eval ===", flush=True)
run_container(
    f"{DOCKER_CMD} {tao_pyt_image} "
    f"clip evaluate -e {cfg.container_path(zs_eval_config_path)}"
)

### Run the DEFT Loop

If you wish to pick up from a previous iteration, modify the `iteration.start` and `iteration.end` parameters in specs/deft_config.yaml.

**Note**: the logs will be long and not very readable. To track the progress, look at the results/ directory. 

In [ ]:
for iter_num in range(cfg.iteration.start, cfg.iteration.end + 1):
    experiment_dir = f"{cfg.base_experiment_path}/iter_{iter_num}"
    training_checkpoint = cfg.training_checkpoint_for_iter(iter_num)

    embed_dir = f"{experiment_dir}/embeddings"
    mining_dir = f"{experiment_dir}/mining"
    vis_dir = f"{experiment_dir}/visualization"
    logs_dir = f"{experiment_dir}/logs"
    gaps_dir = f"{experiment_dir}/gaps"
    gaps_parquet = f"{gaps_dir}/kpi_gaps.parquet"

    # Embedding parquets
    target_embed_pq = f"{embed_dir}/target/embeddings.parquet"
    # For t-SNE visualization
    viz_input_pq = f"{embed_dir}/viz_weak/input.parquet"
    viz_weak_embed_pq = f"{embed_dir}/viz_weak/embeddings.parquet"
    augmented_embed_dir = f"{embed_dir}/augmented"
    mined_embed_pq = f"{augmented_embed_dir}/mined_embeddings.parquet"
    prev_pool_pq = f"{embed_dir}/previous/prev_pool.parquet"
    prev_embed_pq = f"{embed_dir}/previous/embeddings.parquet"

    # Mining outputs
    mined_pq = f"{mining_dir}/mined_samples.parquet"
    # Deduplicated image list fed to the image-embedding container for t-SNE.
    mined_input_pq = f"{mining_dir}/mined_unique_images.parquet"
    mined_image_list_file = f"{mining_dir}/mined_image_list.txt"
    mined_pairs_file = f"{mining_dir}/mined_pairs.json"
    mined_manifest = f"{mining_dir}/mined_dataset.json"
    cumulative_mined_names_file = f"{mining_dir}/cumulative_mined_unique_names.json"

    candidates_dir = (
        f"{mining_dir}/history_candidates"
        if cfg.mining.history_aware.enabled
        else mining_dir
    )
    candidate_image_list_file = f"{candidates_dir}/mined_image_list.txt"
    candidate_pairs_file = f"{candidates_dir}/mined_pairs.json"
    candidate_manifest = f"{candidates_dir}/mined_dataset.json"

    print(f"Iteration {iter_num} experiment dir : {experiment_dir}")
    print(f"Training checkpoint                 : {training_checkpoint}")

    prev_eval_dir = resolve_prev_eval_dir(
        base_experiment_path=cfg.base_experiment_path,
        iter_num=iter_num,
        train_ann_path="",
        eval_subdir="evaluate",
    )

    print(f"\n=== Iter {iter_num}: gap analysis — inference-based (prev eval: {prev_eval_dir}) ===", flush=True)
    analyze_clip_inference_gaps(
        results_dir=prev_eval_dir,
        gaps_parquet=gaps_parquet,
        kpi_image_dir=cfg.pas.eval_image_dir,
        logs_dir=logs_dir,
        kpi_pairs_file=eval_pairs_file,
        metric_name=cfg.gap_analysis.metric_name,
        queries_per_slice=cfg.gap_analysis.queries_per_slice,
        min_num_queries=cfg.gap_analysis.min_num_queries,
        query_types=cfg.gap_analysis.query_types,
        weak_attribute_topk=cfg.gap_analysis.weak_attribute_topk,
        target_query_count=cfg.gap_analysis.target_query_count,
        caption_diversity_enabled=bool_str(cfg.gap_analysis.caption_diversity.enabled),
        caption_history_file=cfg.caption_history_file,
        iter_num=iter_num,
        total_iters=cfg.iteration.end,
        continual_dataset=bool_str(cfg.training.continual_dataset),
        caption_history_policy=cfg.gap_analysis.caption_diversity.history_policy,
        caption_coverage_target=cfg.gap_analysis.caption_diversity.coverage_target,
        min_unique_texts_per_attribute=cfg.gap_analysis.caption_diversity.min_unique_texts_per_attribute,
        max_unique_texts_per_attribute=cfg.gap_analysis.caption_diversity.max_unique_texts_per_attribute,
        max_rows_per_unique_text=cfg.gap_analysis.caption_diversity.max_rows_per_unique_text,
        max_rows_per_image_path=cfg.gap_analysis.caption_diversity.max_rows_per_image_path,
        recent_exclude_iters=cfg.gap_analysis.caption_diversity.recent_exclude_iters,
        replay_fraction_when_noncontinual=cfg.gap_analysis.caption_diversity.replay_fraction_when_noncontinual,
    )

    run_container(
        f"{DOCKER_CMD} {tao_ds_image} "
        f"embedding text_embeddings -e /specs/text_embed_spec.yaml "
        f"input_parquet={cfg.container_path(gaps_parquet)} "
        f"output_parquet={cfg.container_path(target_embed_pq)}"
    )

    run_container(
        f"{DOCKER_CMD} {tao_ds_image} "
        f"tmm nearest_neighbors -e /specs/mining_spec.yaml "
        f"source_parquet={cfg.container_path(source_embed_pq)} "
        f"target_parquet={cfg.container_path(target_embed_pq)} "
        f"output_parquet={cfg.container_path(mined_pq)} "
    )

    print(f"\n=== Iter {iter_num}: k-NN mining summary ===", flush=True)
    summarize_knn_mining(
        mined_parquet=mined_pq,
        target_parquet=target_embed_pq,
        output_dir=mining_dir,
        topn=cfg.mining.topn,
    )

    print(f"\n=== Iter {iter_num}: converting mined parquet to image list ===", flush=True)
    convert_mined_parquet_to_clip_image_list(
        mined_parquet=mined_pq,
        image_dir=cfg.pas.source_image_dir,
        caption_dir=cfg.pas.source_caption_dir,
        caption_file_suffix=".txt",
        output_image_list_file=candidate_image_list_file,
        manifest_path=candidate_manifest,
        source_pairs_file=aug_pool_pairs_file,
        output_pairs_file=candidate_pairs_file,
        # History-aware selection owns the budget and spends it after the
        # novel/replay partition, so the candidate set stays uncapped.
        target_query_count=(
            0 if cfg.mining.history_aware.enabled
            else cfg.gap_analysis.target_query_count
        ),
        caption_expansion_enabled=bool_str(cfg.mining.caption_expansion.enabled),
        caption_expansion_mode=cfg.mining.caption_expansion.mode,
        caption_expansion_max_pairs_per_image_path=cfg.mining.caption_expansion.max_pairs_per_image_path,
        caption_expansion_max_expanded_pair_fraction=cfg.mining.caption_expansion.max_expanded_pair_fraction,
        caption_expansion_dedupe_normalized_caption=bool_str(cfg.mining.caption_expansion.dedupe_normalized_caption),
        caption_expansion_count_expanded_pairs_toward_target=cfg.mining.caption_expansion.count_expanded_pairs_toward_target,
        source_embedding_shards_dir=pool_embed_dir,
        # The mine step emits only `filepath`, so candidate order is its
        # emission order (grouped by target query). Hand it the embeddings
        # it already wrote so the budget keeps the nearest images.
        source_embeddings_parquet=source_embed_pq,
        target_embeddings_parquet=target_embed_pq,
        write_detailed_csv=bool_str(not cfg.mining.history_aware.enabled),
    )

    if cfg.mining.history_aware.enabled:
        print(f"\n=== Iter {iter_num}: history-aware mining selection ===", flush=True)
        select_history_aware_mined_pairs(
            candidate_pairs_file=candidate_pairs_file,
            candidate_manifest_file=candidate_manifest,
            output_image_list_file=mined_image_list_file,
            output_pairs_file=mined_pairs_file,
            manifest_path=mined_manifest,
            history_file=cfg.history_aware_history_file,
            source_pool_image_list_file=aug_pool_image_list_file,
            iter_num=iter_num,
            target_query_count=cfg.gap_analysis.target_query_count,
            continual_dataset=bool_str(cfg.training.continual_dataset),
            replay_fraction=cfg.mining.history_aware.replay_fraction,
            resume="true",
        )
    else:
        print("History-aware mining disabled — using raw KNN output.")

    # Merge this iteration's mined unique_names with the prior iterations' set,
    # so the running total of distinct names mined so far is tracked on disk.
    print(f"\n=== Iter {iter_num}: tracking cumulative mined unique names ===", flush=True)
    track_cumulative_mined_unique_names(
        mined_pairs_file=mined_pairs_file,
        base_experiment_path=cfg.base_experiment_path,
        iter_num=iter_num,
        output_file=cumulative_mined_names_file,
    )

    prev_train_cfg = resolve_prev_clip_train_config(
        base_experiment_path=cfg.base_experiment_path,
        iter_num=iter_num,
        continual_dataset=cfg.training.continual_dataset,
        base_template=cfg.experiment.train_config,
        train_image_list_file="",
    )

    if cfg.visualization.enabled:
        print(f"\n=== Iter {iter_num}: exporting sample contact sheets ===", flush=True)
        export_clip_sample_contact_sheets(
            weak_parquet=gaps_parquet,
            mined_parquet=mined_pq,
            output_dir=f"{vis_dir}/samples",
            source_pairs_file=mined_pairs_file,
            max_samples_per_group=cfg.visualization.max_samples_per_group,
            max_total_samples=cfg.visualization.max_total_samples,
            tile_size=cfg.visualization.tile_size,
        )

    if cfg.visualization.embeddings:
        # Embedding the previous, weak, and mined samples
        prepare_clip_images_for_embedding(
            input_parquet=gaps_parquet,
            output_parquet_path=viz_input_pq,
            image_dir=cfg.pas.eval_image_dir,
        )

        run_container(
            f"{DOCKER_CMD} {tao_ds_image} "
            f"embedding image_embeddings -e /specs/image_embed_spec.yaml "
            f"input_parquet={cfg.container_path(viz_input_pq)} "
            f"output_parquet={cfg.container_path(viz_weak_embed_pq)}"
        )

        prepare_clip_images_for_embedding(
            input_parquet=mined_pq,
            output_parquet_path=mined_input_pq,
            image_dir=cfg.pas.source_image_dir,
        )

        run_container(
            f"{DOCKER_CMD} {tao_ds_image} "
            f"embedding image_embeddings -e /specs/image_embed_spec.yaml "
            f"input_parquet={cfg.container_path(mined_input_pq)} "
            f"output_parquet={cfg.container_path(mined_embed_pq)}"
        )

    _has_prev_train_data = (iter_num > 1) or bool(cfg.pas.train_pairs_source_file)

    if cfg.visualization.embeddings and cfg.training.continual_dataset and _has_prev_train_data:
        print(f"\n=== Iter {iter_num}: preparing previous training data for embedding ===", flush=True)
        # Walks dataset.train.datasets[*] in the prior train config — under
        # continual_dataset that list accumulates one entry per iteration, so it
        # already is the full training history. Its paths are container-absolute,
        # hence the host map for reading the image lists.
        prepare_prev_clip_data_for_embedding(
            prev_train_config_yaml=prev_train_cfg,
            output_parquet_path=prev_pool_pq,
            host_path_map={"/results": HOST_RESULTS_DIR, "/data": HOST_DATA_DIR},
        )

        run_container(
            f"{DOCKER_CMD} {tao_ds_image} "
            f"embedding image_embeddings -e /specs/image_embed_spec.yaml "
            f"input_parquet={cfg.container_path(prev_pool_pq)} "
            f"output_parquet={cfg.container_path(prev_embed_pq)}"
        )
    else:
        print("Skipping previous-training-data embedding (no prior data, or embeddings viz off).")

    if cfg.visualization.embeddings:
        print(f"\n=== Iter {iter_num}: t-SNE visualization ===", flush=True)
        create_tsne_visualization(
            weak_embeddings_dir=f"{embed_dir}/viz_weak",
            augmented_embeddings_dir=augmented_embed_dir,
            previous_embeddings_dir=f"{embed_dir}/previous",
            output_plot_path=f"{vis_dir}/tsne_plot.png",
        )
    else:
        print("cfg.visualization.embeddings=False — skipping t-SNE.")

    print(f"\n=== Iter {iter_num}: building training config ===", flush=True)
    train_config_path = create_clip_train_config(
        base_config_yaml=prev_train_cfg,
        new_config_yaml=f"{experiment_dir}/specs/train_config.yaml",
        output_dir=cfg.container_path(experiment_dir),
        checkpoint_path=training_checkpoint,
        sweep_args=cfg.sweep_args_str,
        mined_image_dir=cfg.pas.source_image_dir,
        mined_caption_dir=cfg.pas.source_caption_dir,
        mined_image_list_file=cfg.container_path(mined_image_list_file),
        caption_file_suffix=".txt",
        train_image_dir=cfg.pas.train_image_dir,
        train_caption_dir=cfg.pas.train_caption_dir,
        train_image_list_file="",
        train_pairs_file="",
        mined_pairs_file=cfg.container_path(mined_pairs_file),
        val_image_list_file=cfg.container_path(val_image_list_file),
        val_image_dir=cfg.pas.eval_image_dir,
        val_caption_dir=cfg.pas.eval_caption_dir,
        continual_dataset=cfg.training.continual_dataset
    )
    print(f"Iter {iter_num} training config: {train_config_path}")

    run_container(
        f"{DOCKER_CMD} {tao_pyt_image} "
        f"clip train -e {cfg.container_path(train_config_path)}"
    )

    # TAO writes only per-epoch checkpoints (plus clip_latest.pth), so select the
    # best one by val/t2i_mAP and publish it at train/best/clip_best_val_t2i_mAP.pth
    # — the path the eval config below expects.
    get_current_checkpoint(f"{experiment_dir}/train")

    # Eval loads the raw Lightning checkpoint above, but the next
    # iteration's train.pretrained_model_path is loaded straight into the
    # SigLIP model and needs the "model." prefix stripped. Write that
    # model-only copy now; training_checkpoint_for_iter points at it.
    normalize_clip_pretrained_checkpoint(
        checkpoint_path=f"{experiment_dir}/train/{cfg.CLIP_CKPT_RELPATH}",
        output_path=f"{experiment_dir}/{cfg.CLIP_PRETRAINED_RELPATH}",
    )

    print(f"\n=== Iter {iter_num}: building eval config ===", flush=True)
    iter_checkpoint = cfg.container_path(f"{experiment_dir}/train/{cfg.CLIP_CKPT_RELPATH}")
    eval_config_path = create_clip_eval_config(
        base_config_yaml=cfg.experiment.eval_config,
        new_config_yaml=f"{experiment_dir}/specs/eval_config.yaml",
        output_dir=cfg.container_path(experiment_dir),
        checkpoint_path=iter_checkpoint,
        eval_image_dir=cfg.pas.eval_image_dir,
        eval_caption_dir=cfg.pas.eval_caption_dir,
        eval_image_list_file=cfg.container_path(eval_image_list_file),
        caption_file_suffix=".txt",
    )
    print(f"Iter {iter_num} eval config: {eval_config_path}")

    run_container(
        f"{DOCKER_CMD} {tao_pyt_image} "
        f"clip evaluate -e {cfg.container_path(eval_config_path)}"
    )

    print(f"\n=== Iter {iter_num}: writing iteration summary ===", flush=True)
    write_iteration_summary(
        experiment_dir=experiment_dir,
        iter_num=iter_num,
        gaps_parquet=gaps_parquet,
        mined_parquet=mined_pq,
        mined_pairs_file=mined_pairs_file,
        training_checkpoint=training_checkpoint,
        next_checkpoint_path=iter_checkpoint,
    )